# Build a Visual Question Answering System with BLIP

This companion notebook contains the complete code for the [To Data & Beyond tutorial](https://todatabeyond.com/blog/building-visual-question-answering-system-using-hugging-face-open-source-models). It uses Hugging Face Transformers and the public `Salesforce/blip-vqa-base` checkpoint to answer a natural-language question about an image.

## Runtime and responsible-use notes

The BLIP checkpoint is roughly 0.4B parameters and downloads model files on first use. A GPU runtime is helpful but not required for this small example. Visual-question-answering models can make confident mistakes, especially with ambiguous, sensitive, or out-of-distribution images; verify answers before using them in consequential settings.

In [ ]:
!pip install -q transformers torch pillow requests

## Load the model and processor

The original article used a machine-local checkpoint path. This notebook uses the official public model ID so it works in a fresh Colab or local environment.

In [ ]:
from pathlib import Path

import requests
from PIL import Image
from transformers import AutoProcessor, BlipForQuestionAnswering

In [ ]:
MODEL_ID = "Salesforce/blip-vqa-base"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = BlipForQuestionAnswering.from_pretrained(MODEL_ID)
model.eval()

## Prepare the tutorial image and question

When opened from GitHub in Colab, the next cell downloads the article's source image from this repository. When the repository has been cloned locally, it uses the checked-in copy instead.

In [ ]:
local_image_path = Path("assets/visual-question-answering/vqa-example.png")
colab_image_path = Path("vqa-example.png")

if local_image_path.exists():
    image_path = local_image_path
else:
    image_url = (
        "https://raw.githubusercontent.com/To-Data-Beyond/"
        "Generative-AI-Techanical-Tutorials/main/assets/"
        "visual-question-answering/vqa-example.png"
    )
    response = requests.get(image_url, timeout=30)
    response.raise_for_status()
    colab_image_path.write_bytes(response.content)
    image_path = colab_image_path

image = Image.open(image_path).convert("RGB")
image

In [ ]:
question = "How many soldiers are in the picture?"

## Run visual question answering

In [ ]:
inputs = processor(images=image, text=question, return_tensors="pt")
{name: tuple(value.shape) for name, value in inputs.items()}

In [ ]:
generated_ids = model.generate(**inputs, max_new_tokens=20)
answer = processor.decode(generated_ids[0], skip_special_tokens=True)
print(answer)

## Reuse the model with another image or question

Pass any Pillow image and question to this helper. Treat its result as a model prediction, not verified ground truth.

In [ ]:
def answer_visual_question(image: Image.Image, question: str) -> str:
    batch = processor(images=image.convert("RGB"), text=question, return_tensors="pt")
    generated_ids = model.generate(**batch, max_new_tokens=20)
    return processor.decode(generated_ids[0], skip_special_tokens=True)